In [ ]:
# =============================================================================
# Scaffold-grouped 10-fold CV: full metric suite + confusion matrices + tables
#   - FP-only vs FP + physchem
#   - PR-AUC = Average Precision (AP)
#   - Threshold metrics computed at threshold=0.5 (editable)
# =============================================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,   # PR-AUC (AP)
    matthews_corrcoef,
    balanced_accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import AllChem, DataStructs
from rdkit.Chem import Descriptors
from rdkit.Chem import Crippen
from rdkit.Chem import rdMolDescriptors

# --------------------------- Paths / I/O -------------------------------------
BASE = r"C:\Users\Besitzer\Desktop\M3_databases"
TRAIN_CSV = os.path.join(BASE, "ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv")
OUTDIR = os.path.join(BASE, "figures", "CV_tables")
os.makedirs(OUTDIR, exist_ok=True)

OUT_CSV_FP  = os.path.join(OUTDIR, "CV10_scaffold_metrics_FPonly.csv")
OUT_CSV_FPP = os.path.join(OUTDIR, "CV10_scaffold_metrics_FPphys.csv")

# --------------------------- Conventions -------------------------------------
TRAIN_LABEL_COL = "consensus_label"
TRAIN_SMILES_PREF = "canonical_smiles"
POS_LABELS = {"active", "active_single"}
NEG_LABELS = {"inactive", "inactive_single"}

FP_RADIUS = 2
FP_BITS = 2048
USE_CHIRALITY = True
USE_FEATURES = False   # Pharmacophore features (RDKit feature definitions) instead of atom invariants

THRESHOLD = 0.5  # decision threshold for Sens/Spec/Prec/F1/confusion matrices

# sample weights (your scheme)
w_map = {"active": 1.0, "inactive": 1.0, "active_single": 0.5, "inactive_single": 0.7}

# --------------------------- Helpers -----------------------------------------
def get_smiles_col(df: pd.DataFrame, preferred: str) -> str:
    if preferred in df.columns:
        return preferred
    for c in ["canonical_smiles", "smiles", "SMILES"]:
        if c in df.columns:
            return c
    raise ValueError("No SMILES column found. Expected one of canonical_smiles/smiles/SMILES.")

def smiles_to_mol(smi: str):
    if not isinstance(smi, str) or not smi.strip():
        return None
    return Chem.MolFromSmiles(smi)

def murcko_scaffold_smiles(mol):
    if mol is None:
        return None
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    if scaf is None or scaf.GetNumAtoms() == 0:
        return None
    return Chem.MolToSmiles(scaf, isomericSmiles=False)

def morgan_fp(mol, radius=FP_RADIUS, n_bits=FP_BITS, use_chirality=USE_CHIRALITY, use_features=USE_FEATURES):
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(
        mol, radius, nBits=n_bits, useChirality=use_chirality, useFeatures=use_features
    )

def fps_to_numpy(bitvect_list):
    """RDKit ExplicitBitVect list -> numpy uint8 (n, n_bits)."""
    n = len(bitvect_list)
    n_bits = bitvect_list[0].GetNumBits()
    X = np.zeros((n, n_bits), dtype=np.uint8)
    for i, bv in enumerate(bitvect_list):
        arr = np.zeros((n_bits,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(bv, arr)
        X[i, :] = arr.astype(np.uint8)
    return X

def compute_physchem(mol):
    """
    Physchem from RDKit (computed HERE):
      MW (Da)             = Descriptors.MolWt(mol)
      logP (XlogP)        = Crippen.MolLogP(mol)
      TPSA (Å^2)          = rdMolDescriptors.CalcTPSA(mol)
      HBD (count)         = rdMolDescriptors.CalcNumHBD(mol)
      HBA (count)         = rdMolDescriptors.CalcNumHBA(mol)
      RotB (count)        = rdMolDescriptors.CalcNumRotatableBonds(mol)
    """
    if mol is None:
        return None
    mw = float(Descriptors.MolWt(mol))
    logp = float(Crippen.MolLogP(mol))
    tpsa = float(rdMolDescriptors.CalcTPSA(mol))
    hbd = float(rdMolDescriptors.CalcNumHBD(mol))
    hba = float(rdMolDescriptors.CalcNumHBA(mol))
    rotb = float(rdMolDescriptors.CalcNumRotatableBonds(mol))
    return np.array([mw, logp, tpsa, hbd, hba, rotb], dtype=np.float32)

def safe_div(a, b):
    return float(a) / float(b) if b else np.nan

def compute_threshold_metrics(y_true, y_prob, threshold=0.5):
    """
    Returns:
      TP, TN, FP, FN, sensitivity, specificity, precision, f1
    """
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()

    sensitivity = safe_div(tp, tp + fn)  # recall / TPR
    specificity = safe_div(tn, tn + fp)  # TNR
    precision  = safe_div(tp, tp + fp)   # PPV
    f1 = 2 * safe_div(precision * sensitivity, precision + sensitivity) if (precision + sensitivity) else np.nan

    return dict(TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                sensitivity=sensitivity, specificity=specificity,
                precision=precision, f1=f1)

def scaffold_cv_full_metrics(pipe, X, y, groups, sample_weight=None, n_splits=10, threshold=0.5, tag="model"):
    """
    Computes per-fold:
      ROC-AUC, PR-AUC(AP), MCC, Balanced Accuracy,
      Sensitivity, Specificity, Precision, F1, confusion matrix counts
    Returns:
      df_folds (per fold), df_summary (mean±sd), totals (aggregate confusion)
    """
    gkf = GroupKFold(n_splits=n_splits)

    rows = []
    # aggregate confusion across folds (not identical to global OOF confusion, but useful)
    agg = dict(TP=0, TN=0, FP=0, FN=0)

    for fold, (tr, te) in enumerate(gkf.split(X, y, groups=groups), start=1):
        X_tr, y_tr = X[tr], y[tr]
        X_te, y_te = X[te], y[te]

        if sample_weight is not None:
            pipe.fit(X_tr, y_tr, clf__sample_weight=sample_weight[tr])
        else:
            pipe.fit(X_tr, y_tr)

        p_te = pipe.predict_proba(X_te)[:, 1]

        # threshold-independent
        roc_auc = roc_auc_score(y_te, p_te)
        pr_auc  = average_precision_score(y_te, p_te)   # PR-AUC (AP)
        # threshold-dependent
        thr = compute_threshold_metrics(y_te, p_te, threshold=threshold)
        y_hat = (p_te >= threshold).astype(int)
        mcc = matthews_corrcoef(y_te, y_hat)
        balacc = balanced_accuracy_score(y_te, y_hat)

        agg["TP"] += thr["TP"]; agg["TN"] += thr["TN"]
        agg["FP"] += thr["FP"]; agg["FN"] += thr["FN"]

        row = dict(
            model=tag,
            fold=fold,
            n_test=int(len(te)),
            pos_test=int(y_te.sum()),
            neg_test=int((y_te == 0).sum()),
            roc_auc=float(roc_auc),
            pr_auc=float(pr_auc),
            mcc=float(mcc),
            bal_acc=float(balacc),
            sensitivity=float(thr["sensitivity"]),
            specificity=float(thr["specificity"]),
            precision=float(thr["precision"]),
            f1=float(thr["f1"]),
            TP=thr["TP"], TN=thr["TN"], FP=thr["FP"], FN=thr["FN"]
        )
        rows.append(row)

        print(f"[{tag} | Fold {fold:02d}] ROC-AUC={roc_auc:.3f} | PR-AUC(AP)={pr_auc:.3f} | "
              f"BalAcc={balacc:.3f} | MCC={mcc:.3f} | "
              f"Sens={thr['sensitivity']:.3f} | Spec={thr['specificity']:.3f} | "
              f"Prec={thr['precision']:.3f} | F1={thr['f1']:.3f}")

    df_folds = pd.DataFrame(rows)

    # mean ± sd table
    metric_cols = ["roc_auc","pr_auc","bal_acc","mcc","sensitivity","specificity","precision","f1"]
    summary = (
        df_folds[metric_cols]
        .agg(["mean","std"])
        .T
        .reset_index()
        .rename(columns={"index":"Metric"})
    )
    summary["Mean±SD"] = summary["mean"].map(lambda x: f"{x:.3f}") + " ± " + summary["std"].map(lambda x: f"{x:.3f}")
    summary = summary[["Metric","Mean±SD","mean","std"]]

    return df_folds, summary, agg

# --------------------------- Load + build features ----------------------------
df = pd.read_csv(TRAIN_CSV)
print("[LOAD]", TRAIN_CSV, "| shape =", df.shape)

smi_col = get_smiles_col(df, TRAIN_SMILES_PREF)

df = df[df[TRAIN_LABEL_COL].isin(POS_LABELS | NEG_LABELS)].copy()
df["y"] = df[TRAIN_LABEL_COL].isin(POS_LABELS).astype(int)
df["w"] = df[TRAIN_LABEL_COL].map(w_map).astype(float)

df["mol"] = df[smi_col].astype(str).map(smiles_to_mol)
df = df[df["mol"].notna()].copy()

df["scaffold"] = df["mol"].map(murcko_scaffold_smiles)
df = df[df["scaffold"].notna()].copy()

# fingerprints
fps = [morgan_fp(m) for m in df["mol"]]
ok_fp = np.array([fp is not None for fp in fps], dtype=bool)
df_fp = df.loc[ok_fp].copy()
fps = [fp for fp in fps if fp is not None]

X_fp = fps_to_numpy(fps).astype(np.float32)
y_fp = df_fp["y"].values.astype(int)
g_fp = df_fp["scaffold"].values
w_fp = df_fp["w"].values.astype(float)

# physchem
phys = [compute_physchem(m) for m in df_fp["mol"]]
ok_phys = np.array([v is not None and np.all(np.isfinite(v)) for v in phys], dtype=bool)

df_fpp = df_fp.loc[ok_phys].copy()
X_fp2 = X_fp[ok_phys]
y_fpp = y_fp[ok_phys]
g_fpp = g_fp[ok_phys]
w_fpp = w_fp[ok_phys]
X_phys = np.vstack([v for v in phys if v is not None and np.all(np.isfinite(v))]).astype(np.float32)

X_fpp = np.hstack([X_fp2, X_phys]).astype(np.float32)

print("[FP-only] n =", len(df_fp), "| pos_frac =", float(y_fp.mean()), "| unique scaffolds =", pd.Series(g_fp).nunique())
print("[FP+phys] n =", len(df_fpp), "| pos_frac =", float(y_fpp.mean()), "| unique scaffolds =", pd.Series(g_fpp).nunique())

# --------------------------- Pipelines ---------------------------------------
pipe_fp = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("clf", LogisticRegression(
        solver="liblinear", max_iter=5000, class_weight="balanced", penalty="l2", C=1.0
    ))
])

pipe_fpp = Pipeline([
    ("scaler", StandardScaler(with_mean=True)),
    ("clf", LogisticRegression(
        solver="liblinear", max_iter=5000, class_weight="balanced", penalty="l2", C=1.0
    ))
])

# --------------------------- Run CV ------------------------------------------
print("\n=== FP-only: full metrics (threshold=%.2f) ===" % THRESHOLD)
df_folds_fp, summary_fp, agg_fp = scaffold_cv_full_metrics(
    pipe_fp, X_fp, y_fp, g_fp, sample_weight=w_fp, n_splits=10, threshold=THRESHOLD, tag="FP-only"
)

print("\n=== FP + physchem: full metrics (threshold=%.2f) ===" % THRESHOLD)
df_folds_fpp, summary_fpp, agg_fpp = scaffold_cv_full_metrics(
    pipe_fpp, X_fpp, y_fpp, g_fpp, sample_weight=w_fpp, n_splits=10, threshold=THRESHOLD, tag="FP+physchem"
)

# --------------------------- Save tables -------------------------------------
df_folds_fp.to_csv(OUT_CSV_FP, index=False)
df_folds_fpp.to_csv(OUT_CSV_FPP, index=False)
print("\n[SAVED]", OUT_CSV_FP)
print("[SAVED]", OUT_CSV_FPP)

print("\n--- Summary (FP-only) ---")
print(summary_fp[["Metric","Mean±SD"]].to_string(index=False))

print("\n--- Summary (FP+physchem) ---")
print(summary_fpp[["Metric","Mean±SD"]].to_string(index=False))

print("\n--- Aggregate confusion across folds (FP-only) ---", agg_fp)
print("--- Aggregate confusion across folds (FP+physchem) ---", agg_fpp)

print("\nDone. Tables saved in:", OUTDIR)


[LOAD] C:\Users\Besitzer\Desktop\M3_databases\ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv | shape = (2268, 13)


[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerator
[23:24:20] DEPRECATION WARNING: please use MorganGenerat

[FP-only] n = 2268 | pos_frac = 0.7883597883597884 | unique scaffolds = 1022
[FP+phys] n = 2268 | pos_frac = 0.7883597883597884 | unique scaffolds = 1022

=== FP-only: full metrics (threshold=0.50) ===
[FP-only | Fold 01] ROC-AUC=0.979 | PR-AUC(AP)=0.994 | BalAcc=0.905 | MCC=0.825 | Sens=0.973 | Spec=0.837 | Prec=0.962 | F1=0.968
[FP-only | Fold 02] ROC-AUC=0.989 | PR-AUC(AP)=0.998 | BalAcc=0.966 | MCC=0.923 | Sens=0.984 | Spec=0.949 | Prec=0.989 | F1=0.987
[FP-only | Fold 03] ROC-AUC=0.980 | PR-AUC(AP)=0.997 | BalAcc=0.873 | MCC=0.724 | Sens=0.960 | Spec=0.786 | Prec=0.970 | F1=0.965
[FP-only | Fold 04] ROC-AUC=0.916 | PR-AUC(AP)=0.984 | BalAcc=0.779 | MCC=0.610 | Sens=0.964 | Spec=0.594 | Prec=0.935 | F1=0.949
[FP-only | Fold 05] ROC-AUC=0.965 | PR-AUC(AP)=0.974 | BalAcc=0.930 | MCC=0.850 | Sens=0.974 | Spec=0.886 | Prec=0.979 | F1=0.977
[FP-only | Fold 06] ROC-AUC=0.942 | PR-AUC(AP)=0.969 | BalAcc=0.871 | MCC=0.772 | Sens=0.960 | Spec=0.782 | Prec=0.894 | F1=0.926
[FP-only | Fold 07

In [2]:
# =============================================================
# Statistical comparison: FP-only vs FP+physchem (paired across folds)
# Using correct column names
# =============================================================

import pandas as pd
import numpy as np
from scipy import stats

fp_path = r"C:\Users\Besitzer\Desktop\M3_databases\figures\CV_tables\CV10_scaffold_metrics_FPonly.csv"
fp_phys_path = r"C:\Users\Besitzer\Desktop\M3_databases\figures\CV_tables\CV10_scaffold_metrics_FPphys.csv"
print(f"[INPUT] FP-only: {fp_path}")
print(f"[INPUT] FP+physchem: {fp_phys_path}")
      
df_fp = pd.read_csv(fp_path).sort_values("fold").reset_index(drop=True)
df_phys = pd.read_csv(fp_phys_path).sort_values("fold").reset_index(drop=True)

metrics = [
    "roc_auc",
    "pr_auc",
    "bal_acc",
    "mcc",
    "sensitivity",
    "specificity",
    "precision",
    "f1"
]

results = []

for m in metrics:
    x = df_fp[m].values
    y = df_phys[m].values
    
    # Paired t-test
    t_stat, p_t = stats.ttest_rel(y, x)
    
    # Wilcoxon signed-rank test (non-parametric alternative)
    try:
        w_stat, p_w = stats.wilcoxon(y, x)
    except:
        w_stat, p_w = np.nan, np.nan
    
    mean_diff = np.mean(y - x)
    
    results.append({
        "Metric": m,
        "Mean_FP": np.mean(x),
        "Mean_FPphys": np.mean(y),
        "Mean_Diff_(FPphys-FP)": mean_diff,
        "Paired_t_p": p_t,
        "Wilcoxon_p": p_w
    })

df_stats = pd.DataFrame(results)

print("Statistical comparison (paired across folds):\n")
print(df_stats.round(6))

out_path = r"C:\Users\Besitzer\Desktop\M3_databases\CV10_statistical_tests_FP_vs_FPphys.csv"
df_stats.to_csv(out_path, index=False)

print(f"[SAVED]: {out_path}")


[INPUT] FP-only: C:\Users\Besitzer\Desktop\M3_databases\figures\CV_tables\CV10_scaffold_metrics_FPonly.csv
[INPUT] FP+physchem: C:\Users\Besitzer\Desktop\M3_databases\figures\CV_tables\CV10_scaffold_metrics_FPphys.csv
Statistical comparison (paired across folds):

        Metric   Mean_FP  Mean_FPphys  Mean_Diff_(FPphys-FP)  Paired_t_p  \
0      roc_auc  0.954742     0.949739              -0.005004    0.582481   
1       pr_auc  0.982612     0.976467              -0.006145    0.218118   
2      bal_acc  0.873016     0.904367               0.031352    0.005708   
3          mcc  0.763038     0.764356               0.001319    0.936205   
4  sensitivity  0.967152     0.934115              -0.033036    0.000207   
5  specificity  0.778880     0.874619               0.095740    0.000544   
6    precision  0.936291     0.960806               0.024515    0.013008   
7           f1  0.950504     0.946582              -0.003922    0.388499   

   Wilcoxon_p  
0    0.695312  
1    0.431641  
2 